In [1]:
import requests
import pandas as pd

BASE_URL = "http://new-service:8000"


In [2]:
# Fetch all accounts
accounts_response = requests.get(f"{BASE_URL}/accounts/", params={"limit": 1000})
accounts_response.raise_for_status()
accounts = accounts_response.json().get("accounts", [])

print(f"{len(accounts)} cuenta(s) encontrada(s)")


4 cuenta(s) encontrada(s)


In [3]:
# Fetch transactions for every account and keep only income
all_transactions = []

for account in accounts:
    account_id = account["id"]
    txn_response = requests.get(f"{BASE_URL}/accounts/{account_id}/transactions")
    txn_response.raise_for_status()
    transactions = txn_response.json().get("transactions", [])
    all_transactions.extend(transactions)

txn_df = pd.DataFrame(all_transactions)
print(f"{len(txn_df)} transacción(es) obtenida(s) en total")


451 transacción(es) obtenida(s) en total


In [4]:
# Filter income and parse dates
income_df = txn_df[txn_df["transaction_type"] == "income"].copy()

income_df["transaction_date"] = pd.to_datetime(
    income_df["transaction_date"], errors="coerce"
)
income_df["amount"] = pd.to_numeric(income_df["amount"], errors="coerce")
income_df = income_df.dropna(subset=["transaction_date", "amount", "account_id"])

income_df = income_df.sort_values(["account_id", "transaction_date"]).reset_index(drop=True)


In [5]:
if income_df.empty:
    print("Sin ingresos registrados.")
else:
    display_df = (
        income_df[["account_id", "transaction_date", "amount", "description"]]
        .copy()
    )
    display_df["transaction_date"] = display_df["transaction_date"].dt.strftime("%d/%m/%Y")

    for account_id, group in display_df.groupby("account_id"):
        print(f"\nCuenta: {account_id}")
        display(
            group.drop(columns="account_id")
            .reset_index(drop=True)
            .style
            .format({"amount": "{:,.2f}"})
            .set_caption(str(account_id))
        )



Cuenta: 191-04106234-1-10


,transaction_date,amount,description
0,29/08/2024,0.00,MANT. CUENTA AGO24
1,31/08/2024,1.95,TRAN.CTAS.PROP.BM
2,15/11/2024,207.90,TRAN.CTAS.PROP.BM
3,29/11/2024,0.00,MANT. CUENTA NOV24
4,15/07/2025,"3,276.19",AB.TR.EXT-AC302390
5,15/07/2025,6.91,TRAN.CTAS.PROP.BM
6,31/07/2025,0.00,MANT. CUENTA JUL25
7,18/08/2025,"4,300.00",AB.TR.EXT-AC656659
8,29/08/2025,85.95,TRAN.CTAS.PROP.BM
9,01/09/2025,0.00,MANT. CUENTA AGO25
